# Wikimedia Event Streaming Pipeline

This notebook defines the `produce(max_events)` function that connects to the [Wikimedia RecentChanges SSE endpoint](https://stream.wikimedia.org/v2/stream/recentchange), collects live edit events, and returns them as a pandas DataFrame. It is designed to be called by the **wikimedia_bronze** notebook (via `%run`), which converts the DataFrame to Spark, persists rows to the bronze Delta table (`workspace.bronze.wikimedia_bronze`), and hands them off to the Silver layer.

## Pipeline Role

1. **Produce** — Connect to the Wikimedia SSE stream, collect up to `max_events` recent-change events, and return them as a pandas DataFrame.
2. **Downstream** — The `wikimedia_bronze` notebook calls `produce()`, converts the result to a Spark DataFrame, and writes it to the bronze Delta table for downstream Silver-layer CDC merge.

## Key Parameters

| Parameter     | Description                                              |
|---------------|---------------------------------------------------------|
| `STREAM_URL`  | Wikimedia SSE endpoint for recent-change events          |
| `max_events`  | Maximum number of events to collect per invocation      |


In [0]:
import json
import logging
import requests
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s")

STREAM_URL = "https://stream.wikimedia.org/v2/stream/recentchange"
HEADERS    = {"User-Agent": "rana-databricks-demo/1.0 (rana@ghazzi.com)"}

def produce(max_events):
    # Read Wikimedia SSE stream, collect events, return as a DataFrame
    resp = requests.get(STREAM_URL, stream=True, timeout=30, headers=HEADERS)
    print("status:", resp.status_code)
    logging.info(f"stream connected, status={resp.status_code}")

    events = []
    for raw_line in resp.iter_lines():
        if not raw_line:
            continue
        decoded = raw_line.decode("utf-8")
        if not decoded.startswith("data: "):
            continue
        payload = decoded[len("data: "):]
        event = json.loads(payload)
        events.append(event)
        if len(events) >= max_events:
            break

    resp.close()
    df = pd.DataFrame(events)
    print(f"[produce] wrote {len(df)} events")
    logging.info(f"produce complete: {len(df)} events collected")
    return df
